# Modeling

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

In [14]:
TRAIN_PATH = Path("../data/processed/train.csv")
VALIDATION_PATH = Path("../data/processed/validation.csv")
TEST_PATH = Path("../data/processed/test.csv")

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df["DateTime"] = pd.to_datetime(train_df["DateTime"])
val_df["DateTime"] = pd.to_datetime(val_df["DateTime"])
test_df["DateTime"] = pd.to_datetime(test_df["DateTime"])

In [15]:
feature_columns = [
    "PT08.S1(CO)", "PT08.S2(NMHC)", "PT08.S3(NOx)",
    "PT08.S4(NO2)", "PT08.S5(O3)", "T", "RH", "AH",
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "day_of_week", "is_weekend",
]

target_columns = ["CO(GT)", "C6H6(GT)", "NOx(GT)", "NO2(GT)"]

In [26]:
split_lengths = {
    "train": len(train_df),
    "validation": len(val_df),
    "test": len(test_df),
}

all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
feature_frame = all_df[["DateTime"] + feature_columns].copy()
missing_before_ffill = feature_frame[feature_columns].isna().sum()
feature_frame[feature_columns] = feature_frame[feature_columns].ffill()
missing_after_ffill = feature_frame[feature_columns].isna().sum()

display(missing_before_ffill.sort_values(ascending=False).to_frame("before_ffill"))
display(missing_after_ffill.sort_values(ascending=False).to_frame("after_ffill"))

,before_ffill
PT08.S1(CO),366
PT08.S2(NMHC),366
PT08.S3(NOx),366
PT08.S4(NO2),366
PT08.S5(O3),366
T,366
RH,366
AH,366
hour_sin,0
hour_cos,0


,after_ffill
PT08.S1(CO),0
PT08.S2(NMHC),0
PT08.S3(NOx),0
PT08.S4(NO2),0
PT08.S5(O3),0
T,0
RH,0
AH,0
hour_sin,0
hour_cos,0


In [31]:
train_features = feature_frame.iloc[:split_lengths["train"]].copy().reset_index(drop=True)
validation_features = feature_frame.iloc[
    split_lengths["train"]:split_lengths["train"] + split_lengths["validation"]
].copy().reset_index(drop=True)
test_features = feature_frame.iloc[
    split_lengths["train"] + split_lengths["validation"]:
].copy().reset_index(drop=True)

X_train_filled = train_features[feature_columns]
X_validation_filled = validation_features[feature_columns]
X_test_filled = test_features[feature_columns]

In [34]:
def make_pipeline(estimator):
    return Pipeline([
        ("median_imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", estimator)
    ])

def make_searches():
    return {
        "Ridge": (
            make_pipeline(Ridge()),
            {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}
        ),
        "RandomForest": (
            make_pipeline(RandomForestRegressor(random_state=42, n_jobs=-1)),
            {
                "model__n_estimators": [200, 400],
                "model__max_depth": [None, 10, 20],
                "model__min_samples_leaf": [1, 2, 5],
                "model__max_features": [1.0, "sqrt"]
            }
        ),
        "XGBoost": (
            make_pipeline(XGBRegressor(
                objective="reg:squarederror",
                eval_metric="rmse",
                random_state=42,
                n_jobs=-1
            )),
            {
                "model__n_estimators": [200, 400],
                "model__learning_rate": [0.03, 0.1],
                "model__max_depth": [3, 6],
                "model__subsample": [0.8, 1.0],
                "model__colsample_bytree": [0.8, 1.0]
            }
        )
    }

def regression_metrics(y_true, y_pred):
    return {
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred)
    }

In [35]:
cv = TimeSeriesSplit(n_splits=4)
primary_scoring = "neg_root_mean_squared_error"

In [37]:
validation_results = []
best_models = {}
validation_predictions = {}

def valid_target_data(features, frame, target):
    mask = frame[target].notna()
    X = features.loc[mask, feature_columns].copy()
    y = frame.loc[mask, target].copy()
    return X, y